In [2]:
import kagglehub
from datasets import load_dataset
import pandas as pd
import dataframes as df
import os

path = kagglehub.dataset_download("ealtman2019/credit-card-transactions")
df_cc = pd.read_csv(f"{path}/credit_card_transactions-ibm_v2.csv")
df_cc.info()
path = kagglehub.dataset_download(
        "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)
print("Path to dataset files:", path)

DATA_DIR = path
print("Using DATA_DIR:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR))

SMALL_TRANS_FILE = os.path.join(DATA_DIR, "HI-Small_Trans.csv")
SMALL_ACCOUNTS_FILE = os.path.join(DATA_DIR,"HI-Small_accounts.csv")
SMALL_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Small_Patterns.txt")

MEDIUM_TRANS_FILE = os.path.join(DATA_DIR, "HI-Medium_Trans.csv")
MEDIUM_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Medium_accounts.csv")
MEDIUM_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Medium_Patterns.txt")

LARGE_TRANS_FILE = os.path.join(DATA_DIR, "HI-Large_Trans.csv")
LARGE_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Large_accounts.csv")
LARGE_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Large_Patterns.txt")

# Raw LI-Small transactions — run Data Analysis on this before preprocessing
df = pd.read_csv(MEDIUM_TRANS_FILE)
display(df.head())
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   User            int64  
 1   Card            int64  
 2   Year            int64  
 3   Month           int64  
 4   Day             int64  
 5   Time            str    
 6   Amount          str    
 7   Use Chip        str    
 8   Merchant Name   int64  
 9   Merchant City   str    
 10  Merchant State  str    
 11  Zip             float64
 12  MCC             int64  
 13  Errors?         str    
 14  Is Fraud?       str    
dtypes: float64(1), int64(7), str(7)
memory usage: 3.7 GB
Path to dataset files: /Users/lachlanbruce/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8
Using DATA_DIR: /Users/lachlanbruce/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8
Files: ['LI-Small_Patterns.txt', 'LI-Medium_Patterns.txt', 'HI-Small_Patte

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0


<class 'pandas.DataFrame'>
RangeIndex: 31898238 entries, 0 to 31898237
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 4.3 GB


In [3]:
# --- Credit card: is any User ID also used as a Merchant Name? ---
users = set(df_cc["User"].unique())
merchants = set(df_cc["Merchant Name"].unique())
user_in_merchant = users & merchants
print("=== df_cc: User vs Merchant Name ===")
print(f"Unique Users:          {len(users):,}")
print(f"Unique Merchant Names: {len(merchants):,}")
print(f"Users also in Merchant Name: {len(user_in_merchant):,}")
print(f"Any User in Merchant Name? {len(user_in_merchant) > 0}")
if user_in_merchant:
    print("Example overlaps:", list(user_in_merchant)[:10])

# --- AML: is any Account also used as Account.1 (receiver)? ---
accounts = set(df["Account"].astype(str).unique())
accounts_1 = set(df["Account.1"].astype(str).unique())
account_in_account1 = accounts & accounts_1

print("\n=== df: Account vs Account.1 ===")
print(f"Unique Account:   {len(accounts):,}")
print(f"Unique Account.1: {len(accounts_1):,}")
print(f"Accounts also in Account.1: {len(account_in_account1):,}")
print(f"Any Account in Account.1? {len(account_in_account1) > 0}")
if account_in_account1:
    print("Example overlaps:", list(account_in_account1)[:10])

# Per-row: sender Account equals receiver Account.1 (self-transfer)
self_transfer = df["Account"].astype(str) == df["Account.1"].astype(str)
print(f"\nRows where Account == Account.1: {self_transfer.sum():,} "
      f"({self_transfer.mean() * 100:.2f}%)")


=== df_cc: User vs Merchant Name ===
Unique Users:          2,000
Unique Merchant Names: 100,343
Users also in Merchant Name: 0
Any User in Merchant Name? False

=== df: Account vs Account.1 ===
Unique Account:   2,013,627
Unique Account.1: 1,689,925
Accounts also in Account.1: 1,626,553
Any Account in Account.1? True
Example overlaps: ['85111F920', '8089C5A60', '80F33B9E0', '82951E480', '8439BF1E0', '838B63B90', '83AFD2E40', '82EACE8D0', '846BE3A10', '817500A30']

Rows where Account == Account.1: 2,561,860 (8.03%)
